# Write your own encoder

The encoder is the only part of a conditional model that changes with the
condition, so it is usually where a new problem starts. This example defines
one from scratch and trains a model with it.

See the [README](../README.md) for setup and the [user guide](../USER_GUIDE.md)
for the complete shape and encoder contracts.

`Encoder` is a structural protocol rather than a base class. Any object that
provides `n_conditions`, `n_qubits`, and three operations works, and
`ConditionalModel` checks the interface when you construct it:

- `sample(key, n_samples)` draws bitstrings from the post-Hadamard reference
  Born law `|psi_c(z)|^2`;
- `amplitude_ratios(samples, masks)` returns `psi_c(z xor a) / psi_c(z)` for
  every Fourier mask `a`;
- `preparation(condition)` returns a callable that queues the PennyLane gates
  preparing `|phi(c)>`, and only the simulation bridge calls it.

Training also requires the encoder to be a JAX pytree whose array fields carry a
leading condition axis. Row `c` of every array field describes condition `c`,
which is how the library restricts the encoder to one condition.

The encoder defined here is the simplest useful one, a Hadamard reference. Its
post-Hadamard state is `|+>^n`, so sampling is uniform bits and every amplitude
ratio is one. All the structure the model fits comes from the learned diagonal,
which makes this a clean check that training works with an encoder the library
has never seen.


## Setup

The budgets in the import cell are small so the notebook runs in seconds. The
fit stays coarse at this size, and the validation MMD still falls over the 100
updates below.


In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import pennylane as qml

import iqp_cond as ciqp

N_QUBITS, N_CONDITIONS = 6, 3
STEPS, BATCH_SIZE, N_OPS, MC_SAMPLES = 100, 64, 32, 1024

## Define the encoder

`preparation` returns a callable that queues the PennyLane gates preparing
`|phi(c)>`. Here it returns a function that queues nothing, because the state
before the first Hadamard layer is `|0>^n` and that layer produces the `|+>^n`
state described by `sample` and `amplitude_ratios`. Training never calls
`preparation`, so a mistake there only shows up when you sample the model.


In [2]:
from collections.abc import Callable
from dataclasses import dataclass


@dataclass(frozen=True)
class HadamardEncoder:
    """Every condition is ``|+>^n``; bits are uniform and ratios are one."""

    # Carries the register shape. This encoder ignores the angle values, but
    # keeping the field makes it a realistic template.
    angles: jnp.ndarray
    name: str = "hadamard"

    @property
    def n_conditions(self) -> int:
        return int(self.angles.shape[0])

    @property
    def n_qubits(self) -> int:
        return int(self.angles.shape[1])

    def sample(self, key: jax.Array, n_samples: int) -> jnp.ndarray:
        # Uniform bits, because the post-Hadamard reference is |+>^n.
        return jax.random.bernoulli(
            key,
            p=0.5,
            shape=(self.n_conditions, n_samples, self.n_qubits),
        ).astype(jnp.float64)

    def amplitude_ratios(self, samples: jnp.ndarray, masks: jnp.ndarray) -> jnp.ndarray:
        # psi(z xor a) / psi(z) is 1 for every sample and mask.
        return jnp.ones((self.n_conditions, samples.shape[1], masks.shape[0]))

    def preparation(self, condition: int) -> Callable[[], None]:
        # |+>^n is the image of |0>^n under the first Hadamard layer, so
        # there is nothing to queue before it.
        return lambda: None

## Register the encoder as a JAX pytree

Training requires this step. `angles` is dynamic JAX state with a leading
condition axis, and `name` is static metadata. The loss indexes the array
fields by condition, so every condition runs in one vectorized program and the
trainer compiles one update for all conditions. An encoder that skips this step
raises a `TypeError` when it trains; the estimator and the simulation bridge
accept either form. `check_encoder` below verifies that restricting the fields
to one row gives that condition's ratios and sampling law.


In [3]:
# Registration returns the node definition, and the binding keeps the cell quiet.
_ = jax.tree_util.register_dataclass(
    HadamardEncoder,
    data_fields=["angles"],
    meta_fields=["name"],
)

## Build a model and its targets

The reference is uniform, so the model has to produce a biased product law whose
bias depends on the condition. Condition `c` gets bit probability
`0.2 + 0.3 * c`, and the targets are drawn straight from that law.


In [4]:
encoder = HadamardEncoder(jnp.zeros((N_CONDITIONS, N_QUBITS)))
model = ciqp.ConditionalModel(ciqp.Ansatz.complete(N_QUBITS), encoder)

rng = np.random.default_rng(0)
# One bit probability per condition, shaped to broadcast over samples and qubits.
probabilities = (0.2 + 0.3 * np.arange(N_CONDITIONS))[:, None, None]
targets = (rng.uniform(size=(N_CONDITIONS, 1000, N_QUBITS)) < probabilities).astype(
    float,
)
validation = targets[:, :200]

print(f"targets {targets.shape}")
print(f"per-condition bit probability {probabilities[:, 0, 0]}")

targets (3, 1000, 6)
per-condition bit probability [0.2 0.5 0.8]


## Train

The configuration looks the same as for a built-in encoder. The improvement
over 100 updates is small, because the budgets are small and the uniform
reference gives the diagonal all the work to do. Raise `STEPS` to push the
validation MMD further down.


In [5]:
result = ciqp.train(
    model,
    targets,
    validation,
    seed=0,
    config=ciqp.TrainingConfig(
        steps=STEPS,
        batch_size=BATCH_SIZE,
        # Masks per update and Monte Carlo reference draws per mask.
        n_ops=N_OPS,
        mc_samples=MC_SAMPLES,
    ),
)
print(f"initial validation MMD {result.validation_losses[0]:.5f}")
print(f"best step {result.best_step}, validation {result.best_validation_loss:.5f}")

initial validation MMD 0.06831
best step 25, validation 0.06828


## Query expectations

`estimate_pauli_z` evaluates Pauli-Z expectations from a fresh reference draw.
The mask `[1] * n` is the full-body observable `Z^n`. Under a product law with
bit probability `p`, its exact value is `(1 - 2 * p) ** n`, which is `+0.047`
for the outer conditions here and `0` for the middle one.

Two effects keep the printed numbers near zero. The fit is short, and the
estimate resamples the reference state, so it carries Monte Carlo error of its
own. Pass `return_samples=True` and read `estimate.standard_errors` to see that
error, or pass `samples=` instead of `key=` to reuse one reference draw across
calls.


In [6]:
estimate = model.estimate_pauli_z(
    result.best_params,
    # One full-body mask over the visible register.
    np.array([[1.0] * N_QUBITS]),
    key=jax.random.PRNGKey(0),
    n_samples=8192,
)
print(f"<Z^{N_QUBITS}> per condition: {np.asarray(estimate.values)[:, 0]}")

<Z^6> per condition: [-0.00433525 -0.01309968 -0.01010907]


## Sample the trained model

Training never touches PennyLane. Sampling builds the circuit from
`preparation`, and `make_qscript` records that callable on its own, which is
the quickest way to check a preparation you wrote by hand.

The per-condition means sit close to 0.5, because the fit is short and the
uniform reference leaves the diagonal all the work. A longer run moves them
toward the target probabilities `0.2`, `0.5`, and `0.8`.


In [7]:
# make_qscript records the preparation without a device. This encoder queues
# nothing, so the list is empty.
script = qml.tape.make_qscript(model.encoder.preparation(0))()
print(f"queued preparation operations {len(script.operations)}")

# Shot samples from the compiled circuit, shaped (conditions, samples, qubits).
draws = model.sample(result.best_params, 256, seed=0)
print(f"draws {draws.shape}")
print(f"mean bit per condition {draws.mean(axis=(1, 2))}")

queued preparation operations 0
draws (3, 256, 6)
mean bit per condition [0.50130208 0.49283854 0.51497396]


## Verify the encoder

`check_encoder` builds the reference state from the encoder's own `preparation`
and compares both classically tractable operations against it, so a sampling or
ratio mistake surfaces here instead of as a quietly biased loss. The notes below
are observations rather than failures: this encoder queues nothing, which is
correct for `|+>^n`, and it does not reject an out-of-range condition.


In [8]:
# Ratios, sampling, preparation, and selection, all against one reference state.
check = ciqp.check_encoder(model.encoder, n_samples=20_000, probes_per_condition=2_048)
print(check.summary())

check_encoder(6 qubits, 3 conditions): ok
  ratios      worst relative error 0.00e+00 over 73728 entries, 0 skipped
  sampling    worst chi-square p-value 4.90e-02 and TV distance 2.49e-02 at 20000 samples
  preparation 0 operations [none], repeat error 0.00e+00
  note: the preparation queues no operations, so phi(c) is |0>^n
  note: preparation does not reject a condition outside the batch


## Where to go next

Raise `STEPS`, `N_OPS`, and `MC_SAMPLES` before reading anything into the
expectation values. At these budgets the printed moments stay far from the
values the product law predicts.

The [user guide](../USER_GUIDE.md) documents the full encoder contract and
training interface. [`train_gaussian.ipynb`](train_gaussian.ipynb) uses the
built-in encoder, while
[`unconditional_fixed_encoder.ipynb`](unconditional_fixed_encoder.ipynb) uses a
fixed reference. Return to the [README](../README.md) for the project overview.
